# Is QAOA worth using for drug selection? — Efficiency benchmark

**Question:** For one fixed roster and budget, can a trained QAOA circuit find a feasible portfolio worth at least **95% of the exact optimum**, with acceptable total computational resources?

This is the **current efficiency experiment**, not the future parameter-transfer study. It preserves the classical dynamic programming, branch-and-bound, and greedy baselines from your original notebooks and uses modern Qiskit Optimization Mapper + `qaoa_ansatz` + `StatevectorEstimator` for QAOA.

Run cells top to bottom with `Dataset.csv`, `classical_utils.py`, and `qaoa_utils.py` next to this notebook. No real QPU account is needed.

In [1]:
# Run once in a new environment; if packages were upgraded, restart kernel.
%pip install -q "qiskit>=2.4,<3" "qiskit-addon-opt-mapper>=0.1,<0.2" numpy pandas scipy matplotlib

Note: you may need to restart the kernel to use updated packages.


## 1. Shared inputs

Change only these settings for a new **single** problem instance. Costs and budget are integer $M. Values are read from the CSV's `Expected value ($M)` column; other CSV columns are preserved. The default budget is 40% of selected costs, rounded down (matching the original benchmark).

In [3]:
import sys
from pathlib import Path

# Find Dataset.csv within the repository
cwd = Path.cwd()
search_roots = [cwd, *cwd.parents]

CSV_PATH = next(
    (csv for root in search_roots
     for csv in root.rglob("Dataset.csv")
     if ".venv" not in csv.parts),
    None
)

if CSV_PATH is None:
    raise FileNotFoundError(
        f"Dataset.csv not found. Working directory: {cwd}"
    )

# Allow importing Python modules alongside the dataset
sys.path.insert(0, str(CSV_PATH.parent))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

from classical_utils import dynamic_programming, branch_and_bound, greedy, portfolio, timed_solver
from qaoa_utils import make_instance, build_qaoa, train_qaoa, evaluate_qaoa, circuit_resources


N_DRUGS = 5
ROW_INDICES = None       # e.g. [0, 2, 5, 9, 12]; None selects the first N_DRUGS
BUDGET = None            # None = floor(0.4 * total selected cost), else integer $M
LAYERS = 1
TRAINING_BUDGETS = (25, 50, 100)  # COBYLA evaluations; independent training runs
TRAINING_SEED = 42
FINAL_SHOTS = 200         # For illustrating finite sampling, NOT for training
TARGET_FRACTION = 0.95
CONFIDENCE = 0.95
MAX_SIM_QUBITS = 20

frame = pd.read_csv(CSV_PATH)
frame.columns = frame.columns.str.strip()
roster = frame.head(N_DRUGS) if ROW_INDICES is None else frame.iloc[ROW_INDICES]
raw_costs = pd.to_numeric(roster["Test Cost ($M)"]).to_numpy()
budget = int(0.4 * sum(raw_costs)) if BUDGET is None else int(BUDGET)
instance = make_instance(roster, budget)
costs, values = instance["costs"], instance["values"]
print(f"Selected {len(costs)} drugs | budget ${budget}M")
display(instance["data"][["Drugs", "Test Cost ($M)", "Expected value ($M)"]])

Selected 5 drugs | budget $12M


,Drugs,Test Cost ($M),Expected value ($M)
0,Drug_1,10,30.09
1,Drug_2,3,24.00
2,Drug_3,4,28.35
3,Drug_4,9,8.70
4,Drug_5,4,11.89


## 2. Exact and heuristic classical baselines

Both **dynamic programming** and **branch-and-bound** solve this integer knapsack exactly; agreement is an important implementation check. **Greedy** is faster but not guaranteed optimal. Median local times are measured over 11 calls (excluding imports and CSV parsing).

In [ ]:
classical_rows = []
solutions = {}
for name, solver in [
    ("Dynamic programming", dynamic_programming),
    ("Branch and bound", branch_and_bound),
    ("Greedy", greedy),
]:
    selected, time_ms = timed_solver(solver, costs, values, budget, repeats=11)
    detail = portfolio(selected, costs, values, instance["names"])
    assert detail["cost_m"] <= budget
    solutions[name] = detail
    classical_rows.append({
        "Method": name,
        "Value ($M)": detail["value_m"],
        "Cost ($M)": detail["cost_m"],
        "Wall time (ms)": time_ms,
        "Selected drugs": detail["drugs"],
    })

optimum = solutions["Dynamic programming"]["value_m"]
assert abs(optimum - solutions["Branch and bound"]["value_m"]) < 1e-7
classical_table = pd.DataFrame(classical_rows)
classical_table["Quality (% of optimum)"] = (
    100 * classical_table["Value ($M)"] / optimum if optimum else 100.0
)
display(classical_table.round(4))
print("Exact optimum ($M):", round(optimum, 2))

## 3. Build QAOA just once

The Qiskit addon builds the constrained problem and its QUBO automatically; Qiskit constructs the circuit. Our utility module retains the converter (to decode auxiliary variables), original Hamiltonian/offset, normalized Hamiltonian, and parameter ordering. Normalization changes the cost-phase angle scale but not the ranking of solutions.

**Current methodology:** COBYLA is trained with *exact ideal statevector expectations*. It does not use finite measurement shots during training. This is an ideal-simulation benchmark, **not** measured QPU runtime.

In [ ]:
model = build_qaoa(instance, layers=LAYERS, qubit_limit=MAX_SIM_QUBITS)
print(f"Qubits: {model['hamiltonian'].num_qubits}; layers: {LAYERS}")
print("Ising terms:", len(model["hamiltonian"]))
print("QUBO penalty:", model["penalty"])
print("Cost-Hamiltonian normalization:", model["scale"])
print("Build time (ms):", round(1000 * model["build_seconds"], 2))
print("Angle order:", [str(param) for param in model["parameters"]])

## 4. Training effort versus final sampling

Train independently at several optimizer budgets, using the **same random initialization** for a controlled comparison. We measure the **exact single-shot probability** of drawing a feasible portfolio worth ≥95% of the optimum, then calculate how many independent final shots would give 95% confidence of seeing such a portfolio at least once.

The best feasible **observed** portfolio, not the most frequent bitstring, is the decision to report. Increasing final shots does not improve the trained state; it only increases opportunities to observe good solutions.

In [ ]:
qaoa_rows = []
training_runs = {}
# Reuse the seed so each training budget starts from the same angles.
for maxiter in TRAINING_BUDGETS:
    fit = train_qaoa(model, maxiter=maxiter, seed=TRAINING_SEED)
    evaluation = evaluate_qaoa(
        model,
        fit["params"],
        optimum,
        target_fraction=TARGET_FRACTION,
        shots=FINAL_SHOTS,
        seed=TRAINING_SEED + maxiter,
    )
    training_runs[maxiter] = (fit, evaluation)
    qaoa_rows.append({
        "Evaluations": fit["evaluations"],
        "Training (s)": fit["training_seconds"],
        "P(feasible)": evaluation["prob_feasible"],
        "P(>=95% optimum)": evaluation["prob_target"],
        "P(exact optimum)": evaluation["prob_optimal"],
        "Final shots for 95% success": evaluation["shots_95"],
        "Best in sampled shots ($M)": evaluation["best_sampled_value"],
        "Best-sampled quality (%)": (
            100 * evaluation["best_sampled_value"] / optimum if optimum else 100
        ),
    })

qaoa_table = pd.DataFrame(qaoa_rows)
display(qaoa_table.round(4))
print("NOTE: Final shots for 95% success is computed from ideal probabilities.")
print("Training used exact expectation values, not finite-shot QPU evaluations.")

## 5. Circuit resources after transpilation

Count logical-basis CX gates and transpiled two-qubit gates/depth on a **synthetic nearest-neighbor line**. This is hardware-aware circuit profiling but **not** execution time or an actual device's fidelity. Transpilation is timed separately; the template is compiled once, not once per shot.

In [ ]:
resources = circuit_resources(model, topology="line")
display(pd.DataFrame([{
    "Qubits": resources["qubits"],
    "Logical CX": resources["logical_cx"],
    "Line-transpiled 2Q gates": resources["transpiled_two_qubit"],
    "Line-transpiled depth": resources["transpiled_depth"],
    "Compilation (ms)": resources["transpile_seconds"] * 1000,
}]).round(3))
print("Native-basis operations:", resources["native_ops"])

## 6. Results and visualizations

The bar chart compares solution quality after the stipulated number of simulated final shots; the line plot shows how the expected **normalized** QUBO energy improved during training. The probabilities are ideal exact-statevector values. A simulator's local wall time is not quantum hardware speed.

In [ ]:
best_budget = max(TRAINING_BUDGETS)
fit, evaluated = training_runs[best_budget]
best_selection = evaluated["best_sampled"]
if best_selection is None:
    print("No feasible portfolio was observed in the finite-shot sample.")
else:
    print("QAOA best feasible observed:", portfolio(best_selection, costs, values, instance["names"]))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4), layout="constrained")
labels = classical_table["Method"].tolist() + [f"QAOA ({best_budget} evals)"]
quality = classical_table["Quality (% of optimum)"].tolist() + [
    100 * evaluated["best_sampled_value"] / optimum if optimum else 100
]
ax1.barh(labels, quality)
ax1.axvline(95, linestyle="--", label="95% target")
ax1.set(xlabel="Best observed portfolio (% of optimum)", xlim=(0, 105))
ax1.legend()
evaluation_numbers = [entry["evaluation"] for entry in fit["history"]]
energies = [entry["energy"] for entry in fit["history"]]
ax2.plot(evaluation_numbers, energies)
ax2.set(
    xlabel="COBYLA objective evaluation",
    ylabel="Mean normalized Ising energy",
    title="Training history",
)
plt.show()

## 7. How to interpret the comparison

**What this notebook can support**

- Same CSV rows, budget, and business valuation across classical and QAOA approaches.
- Exact classical optimum; greedy quality; best feasible portfolio observed from a finite sample.
- Ideal per-shot success probability, and mathematically implied shots required for target confidence.
- Actual local simulation/training wall time, number of optimizer evaluations, mapped qubit count, and synthetic transpiled resource requirements.

**What it cannot support without further work**

- A measured quantum-hardware speedup. `StatevectorEstimator` evaluates expectations exactly, whereas a QPU needs finite-shot estimation during optimization. **Do not claim the total QPU budget is just `FINAL_SHOTS`.**
- Hardware fidelity, execution duration, queues or total cost on a named backend. The synthetic line topology is only a comparison proxy.
- Broad scalability conclusions from this one instance. Classical DP has pseudo-polynomial `O(n * budget)` time for integer weights; ideal statevector simulation scales exponentially with quantum width.

**Efficiency conclusion:** Compare the classical optimum and greedy baseline with the QAOA target-quality probability, training evaluations/time, final shots needed, and transpiled gates. Treat these as separate accounting categories rather than converting circuit gates directly into milliseconds. Parameter-transfer experiments belong on your later branch.

References: [Optimization Mapper](https://qiskit.github.io/qiskit-addon-opt-mapper/) · [Qiskit QAOA tutorial](https://quantum.cloud.ibm.com/docs/en/tutorials/quantum-approximate-optimization-algorithm)